In [2]:
!PREFIX=https://raw.githubusercontent.com/DataTalksClub/llm-zoomcamp/main/02-vector-search/embed && \
curl -O $PREFIX/download.py && \
curl -O $PREFIX/embedder.py

  % Total    % Received % Xferd  Average Speed   Time    Time     Time  Current
                                 Dload  Upload   Total   Spent    Left  Speed
100  1376  100  1376    0     0   4617      0 --:--:-- --:--:-- --:--:--  4632
  % Total    % Received % Xferd  Average Speed   Time    Time     Time  Current
                                 Dload  Upload   Total   Spent    Left  Speed
100  1520  100  1520    0     0  13523      0 --:--:-- --:--:-- --:--:-- 13571


In [6]:
!uv run python download.py

tokenizer.json: 100%|███████████████████████| 712k/712k [00:00<00:00, 12.1MB/s]
  saved models/Xenova/all-MiniLM-L6-v2/tokenizer.json
onnx/model.onnx: 100%|████████████████████| 90.4M/90.4M [00:02<00:00, 33.7MB/s]
  saved models/Xenova/all-MiniLM-L6-v2/model.onnx


#### Question 1

In [7]:
from embedder import Embedder

embed = Embedder()

In [8]:
q1 = "How does approximate nearest neighbor search work?"

v = embed.encode(q1)

print(v.shape)
print(v[0])

(384,)
-0.020582036807885073


#### Question 2

In [9]:
from gitsource import GithubRepositoryDataReader

reader = GithubRepositoryDataReader(
    repo_owner="DataTalksClub",
    repo_name="llm-zoomcamp",
    commit_id="8c1834d",
    allowed_extensions={"md"},
    filename_filter=lambda path: "/lessons/" in path,
)

documents = [file.parse() for file in reader.read()]

In [16]:
target_doc = None

for doc in documents:
    if doc["filename"] == "02-vector-search/lessons/07-sqlitesearch-vector.md":
        target_doc = doc
        break

In [20]:
v2 = embed.encode(target_doc['content'])

In [22]:
v.dot(v2)

np.float64(0.361070280302606)

#### Question 3

In [23]:
from gitsource import chunk_documents
chunks = chunk_documents(documents, size=2000, step=1000)

In [25]:
import numpy as np

# Embed every chunk content
texts = [chunk["content"] for chunk in chunks]

embeddings = embed.encode_batch(texts)
X = np.array(embeddings)

# Embed query
q1 = "How does approximate nearest neighbor search work?"
v = embed.encode(q1)

# If v has shape (1, 384), flatten it
v = v.reshape(-1)

# Score all chunks
scores = X.dot(v)

# Find index of highest score
best_idx = scores.argmax()

# Get best chunk
best_chunk = chunks[best_idx]

best_chunk["filename"]

'02-vector-search/lessons/07-sqlitesearch-vector.md'

#### Question 4

In [42]:
from minsearch import VectorSearch

vindex = VectorSearch(keyword_fields=["course"])
vindex.fit(X, chunks)

In [31]:
query = "What metric do we use to evaluate a search engine?"
query_vector = embed.encode(query)

results = vindex.search(query_vector, num_results=5)

In [33]:
results[0]['filename']

'04-evaluation/lessons/05-search-metrics.md'

#### Question 5

In [35]:
from minsearch import Index
tindex = Index(
    text_fields=["content"],
    keyword_fields=["filename"]
)

tindex.fit(chunks)

In [36]:
query = "How do I store vectors in PostgreSQL?"

vindex = VectorSearch(keyword_fields=["filename"])
vindex.fit(X, chunks)

In [37]:
q_vec = embed.encode(query).reshape(-1)

vector_results = vindex.search(q_vec, num_results=5)

In [38]:
text_results = tindex.search(query, num_results=5)

In [39]:
print("VECTOR RESULTS")
for r in vector_results:
    print(r["filename"], r.get("start"))

print("\nTEXT RESULTS")
for r in text_results:
    print(r["filename"], r.get("start"))

VECTOR RESULTS
02-vector-search/lessons/08-pgvector.md 0
02-vector-search/lessons/08-pgvector.md 3000
03-orchestration/lessons/05-rag.md 2000
02-vector-search/lessons/08-pgvector.md 1000
02-vector-search/lessons/08-pgvector.md 2000

TEXT RESULTS
02-vector-search/lessons/02-embeddings.md 4000
03-orchestration/lessons/05-rag.md 1000
02-vector-search/lessons/01-intro.md 0
03-orchestration/lessons/05-rag.md 0
02-vector-search/lessons/01-intro.md 1000


#### Question 6

In [44]:
def rrf(result_lists, k=60, num_results=5):
    scores = {}
    docs = {}

    for results in result_lists:
        for rank, doc in enumerate(results):
            key = (doc["filename"], doc["start"])
            scores[key] = scores.get(key, 0) + 1 / (k + rank)
            docs[key] = doc

    ranked = sorted(scores, key=scores.get, reverse=True)
    return [docs[key] for key in ranked[:num_results]]

In [48]:
q3 = "How do I give the model access to tools?"

In [49]:
q_vec = embed.encode(q3).reshape(-1)

vector_results = vindex.search(q_vec, num_results=5)

In [50]:
text_results = tindex.search(q3, num_results=5)

In [51]:
results = rrf([vector_results, text_results])

In [52]:
results[0]['filename']

'01-agentic-rag/lessons/13-function-calling.md'